# 06c · Modelado: Propuesta A tal cual el Benchmarking (BO-VMD + RFR-SHAP + SOMA + Bi-NOA-LSTM MIMO)

| | |
|---|---|
| **Fase CRISP-DM** | 4 — Modelado |
| **Objetivo** | Implementar la Propuesta A (1.ª del Benchmarking, 5.00) **tal como la describe la literatura**, en dos variantes MIMO, para compararla con las demás propuestas y con nuestras variantes. |
| **Entradas** | `data/gold/series_filtradas.parquet` (señal BO-VMD, voltaje real, calidad, covariables; solo train), `models/vmd_params.json` |
| **Salidas** | `models/modelo_A1.pt`, `models/modelo_A2.pt`, `reports/modelado/a_*` (RFR-SHAP, SOMA, curvas) |

**Componentes (según `docs/detalle-propuestas-modelos-ml.md`)**
1. **BO-VMD:** la señal filtrada de la capa Gold (notebook 05).
2. **RFR-SHAP:** un Random Forest con valores SHAP decide qué covariables entran al modelo.
3. **Min-Max:** normalización por columna ajustada solo con train; desnormalización inversa para evaluar en voltios.
4. **SOMA:** optimiza tasa de aprendizaje, capas y unidades ocultas.
5. **Bi-NOA-LSTM MIMO:** matriz de entrada (todas las celdas + covariables) → matriz de salida (todas las celdas × 180 días) en un solo pase, con proyección densa final sin activación.

**Dos variantes de MIMO**
- **A1, un modelo por electrolizador:** `[B, 30, N + K]` → `[B, 180, N]` (N = 150 celdas, K = covariables seleccionadas).
- **A2, un modelo conjunto:** `[B, 30, 2N + 2K']` → `[B, 180, 2N]`.

**Tal cual la literatura:** sin máscara de saltos (esa mejora es propia de nuestras variantes). Solo se excluyen del loss los objetivos sin dato válido. Mismo split, misma semilla y el mismo protocolo de early stopping + reentrenamiento que los demás modelos.

**Adaptación al dataset real:** el Benchmarking supone 720 pasos horarios y 183 celdas; aquí son 30 pasos diarios y 150 celdas por electrolizador.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import json
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from voltcast import cleaning as cl
from voltcast import config, io
from voltcast import dataset as ds
from voltcast import features as ft
from voltcast import mimo
from voltcast import model as mdl
from voltcast import soma as sm

config.ensure_dirs()
FIG_DIR = config.REPORTS_DIR / "modelado"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
print(f"Dispositivo: {DEVICE} | SOMA: {config.SOMA} | espacio: {config.SOMA_SPACE}")


def savefig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")

## 1. Datos de train (Gold)

In [ ]:
gold = io.read_layer("gold", "series_filtradas.parquet")
train = gold[gold["split"] == "train"].copy()
train["celda"] = train["celda"].astype(str)
CELLS = sorted(train["celda"].unique())
GRUPOS = {el: [c for c in CELLS if c[0] == el] for el in ("A", "B")}
FECHAS = pd.DatetimeIndex(sorted(train["fecha"].unique()))
T_ = len(FECHAS)
CUT = T_ - config.ES_BLOCK_DAYS

VF = ds.pivot(train, "v_filtrada", CELLS).reindex(FECHAS)          # señal BO-VMD
VR = ds.pivot(train, "voltaje", CELLS).reindex(FECHAS)             # voltaje real (objetivo)
VALIDO = ds.pivot(train, "calidad", CELLS).reindex(FECHAS).isin(cl.CALIDAD_VALIDA)
COV = {el: train[train["celda"] == GRUPOS[el][0]].set_index("fecha")[["kA_z", "vcel_z"]].reindex(FECHAS)
       for el in GRUPOS}
CAL = mimo.calendar_features(FECHAS)
print(f"{len(CELLS)} celdas ({', '.join(f'{el}: {len(c)}' for el, c in GRUPOS.items())}) × {T_} días de train | "
      f"bloque de early stopping desde {FECHAS[CUT].date()}")

## 2. RFR-SHAP: selección de covariables

Candidatas: **kA**, **voltaje medio por celda del electrolizador** (`vcel`), **día del año** y **día de la semana** (seno y coseno), como menciona la estructura de datos del Benchmarking.

Un Random Forest predice el cambio de voltaje a 60 días (`V_real(t0 + 60) − V_filtrada(t0)`) a partir del estado de la celda en `t0` y de las covariables. Usa las 20 celdas de la muestra de BO-VMD y orígenes de train cada 5 días. Las covariables cuya importancia SHAP suma al menos `SHAP_UMBRAL_PCT` % entran al modelo; el voltaje de las celdas entra siempre.

In [ ]:
muestra = [m["celda"] for m in json.loads((config.MODELS_DIR / "vmd_params.json").read_text(encoding="utf-8"))["muestra"]]
muestra = [c for c in muestra if c in CELLS]
filas = []
for c in muestra:
    el = c[0]
    vf, vr, ok = VF[c].to_numpy(), VR[c].to_numpy(), VALIDO[c].to_numpy()
    ka, vc = COV[el]["kA_z"].to_numpy(), COV[el]["vcel_z"].to_numpy()
    for t in range(config.LOOKBACK_DAYS - 1, CUT - 60, 5):
        if not (np.isfinite(vf[t - 29:t + 1]).all() and ok[t + 60]):
            continue
        filas.append({
            "v_t0": vf[t], "pendiente_30d": (vf[t] - vf[t - 29]) / 29,
            "kA_ult": ka[t], "kA_media7": np.nanmean(ka[t - 6:t + 1]),
            "vcel_ult": vc[t], "vcel_media7": np.nanmean(vc[t - 6:t + 1]),
            **CAL.iloc[t].to_dict(), "y": vr[t + 60] - vf[t],
        })
rf_data = pd.DataFrame(filas).dropna()
X_rf, y_rf = rf_data.drop(columns="y"), rf_data["y"]
t = time.time()
imp = ft.rfr_shap(X_rf, y_rf)
GRUPOS_COV = {"kA": ["kA_ult", "kA_media7"], "vcel": ["vcel_ult", "vcel_media7"],
              "doy": ["doy_sin", "doy_cos"], "dow": ["dow_sin", "dow_cos"]}
SELEC = ft.select_covariates(imp, GRUPOS_COV, config.SHAP_UMBRAL_PCT)
print(f"RFR-SHAP sobre {len(rf_data):,} muestras en {time.time() - t:.0f} s | R² (train) del RF: {imp['r2_oob_aprox'][0]:.3f}")
print(f"Covariables seleccionadas (≥ {config.SHAP_UMBRAL_PCT} % SHAP): {SELEC}")
imp.to_csv(FIG_DIR / "a_rfr_shap_importancia.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 3.8))
sns.barplot(data=imp, y="variable", x="porcentaje", ax=ax, color="tab:blue")
ax.set(xlabel="importancia SHAP (% del total)", ylabel="", title="RFR-SHAP: importancia de las variables")
fig.tight_layout(); savefig(fig, "a_rfr_shap")
imp.round(3)

**Lectura.**
- El Random Forest explica ~70 % de la variación del cambio a 60 días en train (R² 0,70). Dominan las variables de la propia celda: su nivel en `t0` (56 %) y su pendiente de 30 días (19 %).
- Entre las covariables, superan el umbral del 5 % **kA** (≈ 14 % entre la media de 7 días y el último valor) y el **día del año** (≈ 6 %). El voltaje medio por celda (≈ 5 %, justo por debajo) y el **día de la semana** (≈ 0 %) quedan fuera.
- Entran al MIMO: las 150 celdas de cada electrolizador + kA + día del año (seno y coseno).

## 3. Matrices MIMO normalizadas (Min-Max, solo train)

In [ ]:
def covariables(el):
    cols = []
    if "kA" in SELEC:
        cols.append(COV[el]["kA_z"].to_numpy())
    if "vcel" in SELEC:
        cols.append(COV[el]["vcel_z"].to_numpy())
    return np.column_stack(cols) if cols else np.zeros((T_, 0))


def calendario():
    cols = [CAL[c].to_numpy() for g in ("doy", "dow") if g in SELEC for c in GRUPOS_COV[g]]
    return np.column_stack(cols) if cols else np.zeros((T_, 0))


ESC, DATOS = {}, {}
for nombre, grupos in [("A1", [["A"], ["B"]]), ("A2", [["A", "B"]])]:
    piezas = {}
    for g in grupos:
        celdas = [c for el in g for c in GRUPOS[el]]
        vr = VR[celdas].where(VALIDO[celdas]).to_numpy()
        cov = np.column_stack([covariables(el) for el in g] + [calendario()])
        csc = mimo.MinMax().fit(vr)
        ksc = mimo.MinMax().fit(cov) if cov.shape[1] else None
        X = mimo.scaled_inputs(VF[celdas].to_numpy(), cov, csc, ksc)
        Y = np.where(VALIDO[celdas].to_numpy(), csc.transform(VR[celdas].to_numpy()), np.nan)
        clave = "".join(g)
        piezas[clave] = (X, Y, VALIDO[celdas].to_numpy(), celdas, csc, ksc)
    DATOS[nombre] = piezas
    for clave, (X, Y, *_ ) in piezas.items():
        print(f"{nombre} · grupo {clave}: entrada [T, {X.shape[1]}] → salida [T, {Y.shape[1]}]")

## 4. SOMA: optimización de hiperparámetros

Espacio de búsqueda: tasa de aprendizaje (`1e-4`–`1e-2`, logarítmica), capas Bi-NOA-LSTM (1–3) y unidades ocultas (32–256). Cada evaluación entrena con early stopping corto sobre el train interno y mide el MSE (normalizado) en el bloque final de train. **Nunca se usa validación.**

- **A1:** SOMA se ejecuta sobre el EL A y el resultado se aplica a ambos electrolizadores (mismo diseño de red).
- **A2:** SOMA propio, sobre el modelo conjunto.

In [ ]:
lo_lr, hi_lr = np.log10(config.SOMA_SPACE["lr"][0]), np.log10(config.SOMA_SPACE["lr"][1])


def decode(u):
    return {"lr": float(10 ** (lo_lr + u[0] * (hi_lr - lo_lr))),
            "layers": int(1 + min(2, int(u[1] * 3))),
            "hidden": int(32 + 16 * round(u[2] * 14))}


def tensores(piezas):
    return ds.MatrixTensors({k: v[0] for k, v in piezas.items()}, {k: v[1] for k, v in piezas.items()},
                            {k: v[2] for k, v in piezas.items()}, FECHAS, DEVICE)


def cfg_de(params, X, Y, **extra):
    return {**config.MIMO_TRAIN, **params, "n_in": X.shape[1], "n_out": Y.shape[1], **extra}


def entrenar(data, cfg, epochs=None, val=True, log=None):
    torch.manual_seed(config.SEED)
    m = mdl.build_model(0, 0, cfg).to(DEVICE)
    vp = data.origins(CUT, T_ - 2) if val else None
    rng = (config.LOOKBACK_DAYS - 1, CUT - 1) if val else (config.LOOKBACK_DAYS - 1, T_ - 2)
    lim = CUT - 1 if val else T_ - 1
    r = mdl.fit(m, data, rng, lim, cfg, val_pairs=vp, val_target_limit=T_ - 1, epochs=epochs,
                mask_jumps=False, log=log)
    return m, r


SOMA_RES = {}
for nombre, clave in [("A1", "A"), ("A2", "AB")]:
    X, Y = DATOS[nombre][clave][0], DATOS[nombre][clave][1]
    data_soma = tensores({clave: DATOS[nombre][clave]})

    def objetivo(params):
        cfg = cfg_de(params, X, Y, max_epochs=config.SOMA["max_epochs"], patience=config.SOMA["patience"])
        return entrenar(data_soma, cfg)[1]["mejor_val_mse"]

    print(f"\n=== SOMA {nombre} (grupo {clave}) ===")
    t = time.time()
    SOMA_RES[nombre] = sm.soma(objetivo, decode, dim=3, pop=config.SOMA["pop"], migrations=config.SOMA["migrations"],
                               path_length=config.SOMA["path_length"], step=config.SOMA["step"],
                               prt=config.SOMA["prt"], seed=config.SEED, log=print)
    SOMA_RES[nombre]["minutos"] = (time.time() - t) / 60
    print(f"→ {nombre}: mejor {SOMA_RES[nombre]['mejor']} (MSE bloque {SOMA_RES[nombre]['objetivo']:.5f}), "
          f"{SOMA_RES[nombre]['evaluaciones']} evaluaciones en {SOMA_RES[nombre]['minutos']:.0f} min")
    pd.DataFrame(SOMA_RES[nombre]["historial"]).to_csv(FIG_DIR / f"a_soma_{nombre}.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
for ax, (nombre, r) in zip(axes, SOMA_RES.items()):
    h = pd.DataFrame(r["historial"])
    ax.scatter(range(len(h)), h["objetivo"], c=h["migracion"], cmap="viridis", s=18)
    ax.plot(range(len(h)), h["objetivo"].cummin(), color="k", lw=1, label="mejor hasta el momento")
    ax.set(yscale="log", xlabel="evaluación", ylabel="MSE bloque interno (normalizado)",
           title=f"SOMA {nombre}: mejor {r['mejor']}")
    ax.legend(fontsize=8)
fig.tight_layout(); savefig(fig, "a_soma")

**Lectura.**
- **A1** (EL A): 65 evaluaciones en 73 min. Mejor configuración: lr ≈ 0,0044, **2 capas, 176 unidades** (MSE del bloque 0,0053).
- **A2** (conjunto): 65 evaluaciones en 104 min. Mejor configuración: **lr = 0,01, 1 capa, 256 unidades** (0,0052), en el **borde superior** del espacio de búsqueda para la tasa de aprendizaje y las unidades: un espacio más amplio podría encontrar algo mejor, aunque la mejora entre las mejores configuraciones es pequeña.
- La superficie es poco selectiva: muchas configuraciones distintas quedan cerca del óptimo (el peor resultado es ~2 veces el mejor).

## 5. Entrenamiento final (early stopping + reentrenamiento con todo train)

In [ ]:
FINAL, HIST, TIEMPO = {}, [], {}
for nombre in ("A1", "A2"):
    params = SOMA_RES[nombre]["mejor"]
    t_total, modelos = 0.0, {}
    for clave, piezas in DATOS[nombre].items():
        data = tensores({clave: piezas})
        cfg = cfg_de(params, piezas[0], piezas[1])
        t = time.time()
        m_es, r_es = entrenar(data, cfg, log=None)
        m_fin, r_fin = entrenar(data, cfg, epochs=r_es["mejor_epoca"], val=False)
        t_total += time.time() - t
        modelos[clave] = (m_fin, cfg, r_es)
        HIST += [{"modelo": nombre, "grupo": clave, "fase": "early_stopping", **h} for h in r_es["historia"]]
        HIST += [{"modelo": nombre, "grupo": clave, "fase": "reentrenamiento", **h} for h in r_fin["historia"]]
        print(f"{nombre} · {clave}: mejor época {r_es['mejor_epoca']}, MSE bloque {r_es['mejor_val_mse']:.5f}, "
              f"{mdl.count_params(m_fin):,} parámetros")
    FINAL[nombre] = modelos
    TIEMPO[nombre] = {"soma_min": SOMA_RES[nombre]["minutos"], "entrenamiento_min": t_total / 60}
    print(f"→ {nombre}: entrenamiento final {t_total / 60:.1f} min (+ SOMA {SOMA_RES[nombre]['minutos']:.0f} min)")
pd.DataFrame(HIST).to_csv(FIG_DIR / "a_historial_entrenamiento.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
h = pd.DataFrame(HIST)
for (nombre, grupo), g in h[h["fase"] == "early_stopping"].groupby(["modelo", "grupo"]):
    l, = ax.plot(g["epoca"], g["train_mse"], label=f"{nombre}·{grupo} train")
    ax.plot(g["epoca"], g["val_mse"], ls="--", color=l.get_color(), label=f"{nombre}·{grupo} bloque interno")
ax.set(yscale="log", xlabel="época", ylabel="MSE (normalizado)", title="Propuesta A: curvas de entrenamiento")
ax.legend(fontsize=7, ncol=2); fig.tight_layout(); savefig(fig, "a_curvas")

## 6. Guardado

In [ ]:
for nombre, modelos in FINAL.items():
    grupos = list(modelos)
    red = mdl.GroupModels({g: modelos[g][0] for g in grupos})
    meta = {
        "nombre": f"Propuesta A ({'MIMO por electrolizador' if nombre == 'A1' else 'MIMO conjunto'})",
        "arch": "mimo", "variante": nombre, "grupos": grupos,
        "celdas_por_grupo": {g: DATOS[nombre][g][3] for g in grupos},
        "train_cfg": {**modelos[grupos[0]][1], "groups": grupos} if len(grupos) > 1 else modelos[grupos[0]][1],
        "cfg_por_grupo": {g: modelos[g][1] for g in grupos},
        "escaladores": {g: {"celdas": DATOS[nombre][g][4].to_dict(),
                            "covariables": DATOS[nombre][g][5].to_dict() if DATOS[nombre][g][5] else None} for g in grupos},
        "covariables": SELEC, "senal_entrada": "v_filtrada (BO-VMD)", "soma": {k: v for k, v in SOMA_RES[nombre].items() if k != "historial"},
        "mejor_epoca": {g: modelos[g][2]["mejor_epoca"] for g in grupos}, "tiempo_min": TIEMPO[nombre],
        "lookback": config.LOOKBACK_DAYS, "horizon": config.HORIZON_DAYS, "seed": config.SEED,
        "mask_post_jump": False, "entrenado_hasta": str(FECHAS[-1].date()),
        "cells": CELLS, "features": [], "use_noa": True,
        "fecha_entrenamiento": pd.Timestamp.now().isoformat(timespec="seconds"),
    }
    path = config.MODELS_DIR / f"modelo_{nombre}.pt"
    mdl.save_checkpoint(path, red, meta)
    print(f"✔ {path.relative_to(config.BASE_DIR)} ({path.stat().st_size / 1e6:.1f} MB)")

## 7. Verificaciones

In [ ]:
checks = {}
for nombre, modelos in FINAL.items():
    ck = torch.load(config.MODELS_DIR / f"modelo_{nombre}.pt", map_location=DEVICE, weights_only=False)
    red = mdl.GroupModels({g: mdl.build_model(0, 0, ck["cfg_por_grupo"][g]) for g in ck["grupos"]}).to(DEVICE)
    red.load_state_dict(ck["state_dict"]); red.eval()
    for i, g in enumerate(ck["grupos"]):
        x = torch.as_tensor(DATOS[nombre][g][0][None, -config.LOOKBACK_DAYS:], device=DEVICE)
        cell = torch.full((1,), i, device=DEVICE)
        with torch.no_grad():
            p1, p2 = red(x, cell), modelos[g][0].eval()(x)
        checks[f"{nombre}·{g}: el checkpoint reproduce las predicciones"] = torch.allclose(p1, p2, atol=1e-5)
        checks[f"{nombre}·{g}: salida [1, {config.HORIZON_DAYS}, {len(DATOS[nombre][g][3])}]"] = tuple(p1.shape) == (1, config.HORIZON_DAYS, len(DATOS[nombre][g][3]))
checks["solo fechas de train"] = FECHAS[-1] < gold.loc[gold["split"] == "val", "fecha"].min()
for k, v in checks.items():
    print(("✔" if v else "✘"), k)
assert all(checks.values())

## Conclusión

- Se implementó la **Propuesta A tal cual el Benchmarking**: BO-VMD, RFR-SHAP (kA y día del año), Min-Max, SOMA y Bi-NOA-LSTM MIMO con salida directa, en dos variantes (A1 por electrolizador y A2 conjunto), sin máscara de saltos.
- Costo total ≈ 3 h, casi todo en SOMA (65 entrenamientos cortos por variante).
- Salidas: `models/modelo_A1.pt` y `models/modelo_A2.pt` (con escaladores, covariables seleccionadas y resultado de SOMA). La comparación con los demás modelos está en el notebook 07.